In [1]:
import pandas as pd
import numpy as np

# 1. Load the messy dataset
df = pd.read_csv('teaching_prices_messy.csv')

print("--- INITIAL INSPECTION ---")
print(f"Shape: {df.shape}")
print(df.info())

# 2. Drop exact duplicate rows (specifically Row 96, duplicate of Row 8)
df = df.drop_duplicates().reset_index(drop=True)

# 3. Clean price column: strip commas and convert to numeric float, coercing errors
df['price_cleaned'] = pd.to_numeric(
    df['price'].astype(str).str.replace(',', ''), errors='coerce'
)

# 4. Handle specific data issues identified in the audit:
# - Row 40 (originally index 40): Item mislabeled as 'oil b' with price 108.5 -> Correct to 'Staple A'
df.loc[(df['item'].str.lower() == 'oil b') & (df['date'] == '2026-02-05') & (df['city'] == 'City 1') & (df['price_cleaned'] == 108.5), 'item'] = 'Staple A'

# - Row 30: Sentinel value 9999 for Sweetener D -> Convert to NaN
df.loc[df['price_cleaned'] == 9999, 'price_cleaned'] = np.nan

# 5. Add audit flags for transparency instead of silent deletion
df['flag_outlier'] = False
# Flag extreme outlier (e.g., 1250.0 for Staple A)
df.loc[df['price_cleaned'] > 1000, 'flag_outlier'] = True

df['flag_missing'] = df['price_cleaned'].isna()

print("\n--- CLEANED DATA SUMMARY ---")
print(f"Final shape: {df.shape}")
print("Item value counts:")
print(df['item'].value_counts())
print("\nMissing or flagged values count:")
print(df[['flag_missing', 'flag_outlier']].sum())

# 6. Save the audited clean file
df.to_csv('teaching_prices_audited_clean.csv', index=False)
print("\nAudited clean file successfully saved as 'teaching_prices_audited_clean.csv'.")

--- INITIAL INSPECTION ---
Shape: (97, 6)
<class 'pandas.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        97 non-null     str  
 1   item        97 non-null     str  
 2   city        97 non-null     str  
 3   price       96 non-null     str  
 4   unit        97 non-null     str  
 5   data_label  97 non-null     str  
dtypes: str(6)
memory usage: 15.1 KB
None

--- CLEANED DATA SUMMARY ---
Final shape: (96, 9)
Item value counts:
item
Staple A       24
Oil B          24
Pulse C        24
Sweetener D    24
Name: count, dtype: int64

Missing or flagged values count:
flag_missing    2
flag_outlier    1
dtype: int64

Audited clean file successfully saved as 'teaching_prices_audited_clean.csv'.


Line-by-Line Verdict & Evidence:
Line 1 (drop_duplicates()):Verdict: KEEPEvidence: Successfully removes Row 96 (exact duplicate of Row 8). Drops row count from 97 to 96. No valid unique observations were lost.
Line 2 (str.replace(',', '') & to_numeric()):Verdict: MODIFYEvidence: Essential for type conversion (object $\rightarrow$ float64), but must be done carefully so missing values aren't blindly converted without tracking flags.
Line 3 (dropna(subset=['price'])):Verdict: REJECTEvidence: Silently deletes Row 5 (Pulse C, City 2, 2026-01-01). This reduces the row count for 2026-01-01 from 8 to 7, breaking panel balance. The missing observation should be retained with NaN and flagged.
Line 4 (df[df['price'] < 1000]):Verdict: REJECTEvidence: Silently drops Row 17 (1250.0) and Row 30 (9999). While these values are erroneous, silently deleting rows destroys audit visibility. They should be retained and flagged in an audit column (flag_outlier), or set to NaN with explicit documentation.
Line 5 (df['item'] = df['item'].str.title()):Verdict: MODIFY / REJECT AS STANDALONEEvidence: Running .str.title() changes oil b to Oil B. This creates two Oil B records for City 1 on 2026-02-05 (Row 40 with price 108.5 and Row 42 with price 318.0), 
while leaving Staple A missing! The AI blindly renamed strings without inspecting price context. Row 40 must be explicitly corrected to Staple A.